# Notebook 07: Training and Text Generation

The final stage. Steps:
1. Build an **Adam optimizer** from scratch
2. **Train** the transformer on the Shakespeare text
3. Track and plot **training loss**
4. **Generate text** from the trained model
5. Observe how the model learns character patterns

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
np.random.seed(42)

## 1. All Model Components

We include all the components from previous notebooks here so this notebook runs independently.

In [ ]:
# === Utility functions ===

def softmax(x):
    e_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e_x / np.sum(e_x, axis=-1, keepdims=True)


def cross_entropy_loss(logits, targets):
    N = logits.shape[0]
    probs = softmax(logits)
    log_probs = -np.log(probs[np.arange(N), targets] + 1e-9)
    loss = np.mean(log_probs)
    dlogits = probs.copy()
    dlogits[np.arange(N), targets] -= 1
    dlogits /= N
    return loss, dlogits


def causal_mask(seq_len):
    mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
    return mask[np.newaxis, np.newaxis, :, :]


def get_positional_encoding(max_seq_len, d_model):
    pe = np.zeros((max_seq_len, d_model))
    position = np.arange(max_seq_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe


def get_batch(data, batch_size, seq_len):
    max_start = len(data) - seq_len - 1
    starts = np.random.randint(0, max_start, size=batch_size)
    x = np.array([data[s:s+seq_len] for s in starts])
    y = np.array([data[s+1:s+seq_len+1] for s in starts])
    return x, y

In [ ]:
# === Model components ===

class Embedding:
    def __init__(self, vocab_size, d_model):
        self.W = np.random.randn(vocab_size, d_model) * 0.02
        self.dW = None
        self.indices = None
    
    def forward(self, indices):
        self.indices = indices
        return self.W[indices]
    
    def backward(self, dout):
        self.dW = np.zeros_like(self.W)
        np.add.at(self.dW, self.indices, dout)


class LayerNorm:
    def __init__(self, d_model, eps=1e-5):
        self.gamma = np.ones(d_model)
        self.beta = np.zeros(d_model)
        self.eps = eps
        self.dgamma = None
        self.dbeta = None
        self.x_hat = None
        self.std_inv = None
    
    def forward(self, x):
        mean = np.mean(x, axis=-1, keepdims=True)
        var = np.var(x, axis=-1, keepdims=True)
        self.std_inv = 1.0 / np.sqrt(var + self.eps)
        self.x_hat = (x - mean) * self.std_inv
        return self.gamma * self.x_hat + self.beta
    
    def backward(self, dout):
        D = dout.shape[-1]
        dout_flat = dout.reshape(-1, D)
        x_hat_flat = self.x_hat.reshape(-1, D)
        self.dgamma = np.sum(dout_flat * x_hat_flat, axis=0)
        self.dbeta = np.sum(dout_flat, axis=0)
        dx_hat = dout * self.gamma
        dx = self.std_inv * (
            dx_hat
            - np.mean(dx_hat, axis=-1, keepdims=True)
            - self.x_hat * np.mean(dx_hat * self.x_hat, axis=-1, keepdims=True)
        )
        return dx


class MultiHeadAttention:
    def __init__(self, d_model, n_heads):
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        scale = np.sqrt(2.0 / (d_model + self.d_k))
        self.W_Q = np.random.randn(d_model, d_model) * scale
        self.W_K = np.random.randn(d_model, d_model) * scale
        self.W_V = np.random.randn(d_model, d_model) * scale
        self.W_O = np.random.randn(d_model, d_model) * scale
        self.dW_Q = self.dW_K = self.dW_V = self.dW_O = None
        self.x = self.Q = self.K = self.V = None
        self.attn_weights = self.attn_output = None
    
    def _split_heads(self, x):
        B, T, D = x.shape
        return x.reshape(B, T, self.n_heads, self.d_k).transpose(0, 2, 1, 3)
    
    def _merge_heads(self, x):
        B, H, T, d_k = x.shape
        return x.transpose(0, 2, 1, 3).reshape(B, T, self.d_model)
    
    def forward(self, x, mask=None):
        self.x = x
        Q = x @ self.W_Q
        K = x @ self.W_K
        V = x @ self.W_V
        self.Q = self._split_heads(Q)
        self.K = self._split_heads(K)
        self.V = self._split_heads(V)
        scores = (self.Q @ self.K.transpose(0, 1, 3, 2)) / np.sqrt(self.d_k)
        if mask is not None:
            scores = np.where(mask, -1e9, scores)
        self.attn_weights = softmax(scores)
        attn_out = self.attn_weights @ self.V
        self.attn_output = self._merge_heads(attn_out)
        return self.attn_output @ self.W_O
    
    def backward(self, dout):
        B, T, D = dout.shape
        self.dW_O = self.attn_output.reshape(-1, D).T @ dout.reshape(-1, D)
        d_attn_output = dout @ self.W_O.T
        d_attn_out = self._split_heads(d_attn_output)
        d_attn_weights = d_attn_out @ self.V.transpose(0, 1, 3, 2)
        dV = self.attn_weights.transpose(0, 1, 3, 2) @ d_attn_out
        sum_term = np.sum(d_attn_weights * self.attn_weights, axis=-1, keepdims=True)
        d_scores = self.attn_weights * (d_attn_weights - sum_term)
        d_scores /= np.sqrt(self.d_k)
        dQ = d_scores @ self.K
        dK = d_scores.transpose(0, 1, 3, 2) @ self.Q
        dQ = self._merge_heads(dQ)
        dK = self._merge_heads(dK)
        dV = self._merge_heads(dV)
        x_flat = self.x.reshape(-1, D)
        self.dW_Q = x_flat.T @ dQ.reshape(-1, D)
        self.dW_K = x_flat.T @ dK.reshape(-1, D)
        self.dW_V = x_flat.T @ dV.reshape(-1, D)
        dx = dQ @ self.W_Q.T + dK @ self.W_K.T + dV @ self.W_V.T
        return dx


class FeedForward:
    def __init__(self, d_model, d_ff):
        scale1 = np.sqrt(2.0 / (d_model + d_ff))
        scale2 = np.sqrt(2.0 / (d_ff + d_model))
        self.W1 = np.random.randn(d_model, d_ff) * scale1
        self.b1 = np.zeros(d_ff)
        self.W2 = np.random.randn(d_ff, d_model) * scale2
        self.b2 = np.zeros(d_model)
        self.dW1 = self.db1 = self.dW2 = self.db2 = None
        self.x = self.hidden_relu = self.relu_mask = None
    
    def forward(self, x):
        self.x = x
        hidden = x @ self.W1 + self.b1
        self.relu_mask = (hidden > 0).astype(float)
        self.hidden_relu = hidden * self.relu_mask
        return self.hidden_relu @ self.W2 + self.b2
    
    def backward(self, dout):
        d_model = dout.shape[-1]
        d_ff = self.W1.shape[1]
        self.dW2 = self.hidden_relu.reshape(-1, d_ff).T @ dout.reshape(-1, d_model)
        self.db2 = dout.reshape(-1, d_model).sum(axis=0)
        d_hidden = (dout @ self.W2.T) * self.relu_mask
        self.dW1 = self.x.reshape(-1, d_model).T @ d_hidden.reshape(-1, d_ff)
        self.db1 = d_hidden.reshape(-1, d_ff).sum(axis=0)
        return d_hidden @ self.W1.T


class Linear:
    def __init__(self, in_features, out_features):
        scale = np.sqrt(2.0 / (in_features + out_features))
        self.W = np.random.randn(in_features, out_features) * scale
        self.b = np.zeros(out_features)
        self.dW = self.db = None
        self.x = None
    
    def forward(self, x):
        self.x = x
        return x @ self.W + self.b
    
    def backward(self, dout):
        self.dW = self.x.reshape(-1, self.x.shape[-1]).T @ dout.reshape(-1, dout.shape[-1])
        self.db = dout.reshape(-1, dout.shape[-1]).sum(axis=0)
        return dout @ self.W.T


class TransformerBlock:
    def __init__(self, d_model, n_heads, d_ff):
        self.ln1 = LayerNorm(d_model)
        self.mha = MultiHeadAttention(d_model, n_heads)
        self.ln2 = LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
        self.x = None
        self.x_after_attn = None
    
    def forward(self, x, mask=None):
        self.x = x
        x_norm = self.ln1.forward(x)
        attn_out = self.mha.forward(x_norm, mask=mask)
        self.x_after_attn = x + attn_out
        x_norm2 = self.ln2.forward(self.x_after_attn)
        ffn_out = self.ffn.forward(x_norm2)
        return self.x_after_attn + ffn_out
    
    def backward(self, dout):
        d_ffn_out = dout
        d_x_norm2 = self.ffn.backward(d_ffn_out)
        d_x_after_attn = dout + self.ln2.backward(d_x_norm2)
        d_attn_out = d_x_after_attn
        d_x_norm = self.mha.backward(d_attn_out)
        dx = d_x_after_attn + self.ln1.backward(d_x_norm)
        return dx
    
    def get_params(self):
        return [
            (self.ln1, 'gamma'), (self.ln1, 'beta'),
            (self.mha, 'W_Q'), (self.mha, 'W_K'),
            (self.mha, 'W_V'), (self.mha, 'W_O'),
            (self.ln2, 'gamma'), (self.ln2, 'beta'),
            (self.ffn, 'W1'), (self.ffn, 'b1'),
            (self.ffn, 'W2'), (self.ffn, 'b2'),
        ]


class Transformer:
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_seq_len):
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        self.embedding = Embedding(vocab_size, d_model)
        self.pe = get_positional_encoding(max_seq_len, d_model)
        self.blocks = [TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)]
        self.ln_final = LayerNorm(d_model)
        self.output_proj = Linear(d_model, vocab_size)
    
    def forward(self, indices):
        B, T = indices.shape
        x = self.embedding.forward(indices) + self.pe[:T]
        mask = causal_mask(T)
        for block in self.blocks:
            x = block.forward(x, mask=mask)
        x = self.ln_final.forward(x)
        return self.output_proj.forward(x)
    
    def backward(self, dlogits):
        dx = self.output_proj.backward(dlogits)
        dx = self.ln_final.backward(dx)
        for block in reversed(self.blocks):
            dx = block.backward(dx)
        self.embedding.backward(dx)
    
    def get_params(self):
        params = [(self.embedding, 'W')]
        for block in self.blocks:
            params.extend(block.get_params())
        params.extend([
            (self.ln_final, 'gamma'), (self.ln_final, 'beta'),
            (self.output_proj, 'W'), (self.output_proj, 'b'),
        ])
        return params
    
    def count_params(self):
        return sum(getattr(obj, name).size for obj, name in self.get_params())

print("All model components defined.")

## 2. Adam Optimizer

Adam combines momentum and adaptive learning rates:

$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
$$\theta_t = \theta_{t-1} - \alpha \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

In [ ]:
class Adam:
    """Adam optimizer."""
    
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        """
        Args:
            params: list of (object, param_name) tuples
            lr: learning rate
        """
        self.params = params
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0
        
        # Initialize moment estimates
        self.m = {}  # first moment (mean of gradients)
        self.v = {}  # second moment (mean of squared gradients)
        for i, (obj, name) in enumerate(params):
            param = getattr(obj, name)
            self.m[i] = np.zeros_like(param)
            self.v[i] = np.zeros_like(param)
    
    def step(self):
        """Update all parameters."""
        self.t += 1
        for i, (obj, name) in enumerate(self.params):
            param = getattr(obj, name)
            grad = getattr(obj, 'd' + name)
            
            if grad is None:
                continue
            
            # Clip gradients to prevent explosion
            grad = np.clip(grad, -1.0, 1.0)
            
            # Update moments
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * grad
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * grad**2
            
            # Bias correction
            m_hat = self.m[i] / (1 - self.beta1**self.t)
            v_hat = self.v[i] / (1 - self.beta2**self.t)
            
            # Update parameters
            param -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
            setattr(obj, name, param)

print("Adam optimizer defined.")

## 3. Load Data

In [ ]:
# Load and prepare data
with open('../data/input.txt', 'r') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [char_to_idx[c] for c in s]

def decode(indices):
    return ''.join([idx_to_char[i] for i in indices])

data = np.array(encode(text), dtype=np.int64)
print(f"Text length: {len(text)} characters")
print(f"Vocabulary size: {vocab_size}")
print(f"Characters: {''.join(chars)}")

## 4. Text Generation Function

Autoregressive generation: predict one character at a time, feeding each prediction back as input.

In [ ]:
def generate(model, start_str, max_len=200, temperature=0.8):
    """Generate text autoregressively.
    
    Args:
        model: trained Transformer
        start_str: seed string to start generation
        max_len: number of characters to generate
        temperature: controls randomness (lower = more deterministic)
    """
    indices = encode(start_str)
    
    for _ in range(max_len):
        # Take last max_seq_len tokens as context
        context = indices[-model.max_seq_len:]
        x = np.array([context])  # (1, T)
        
        # Forward pass
        logits = model.forward(x)  # (1, T, vocab_size)
        
        # Get logits for the last position
        next_logits = logits[0, -1, :]  # (vocab_size,)
        
        # Apply temperature
        next_logits = next_logits / temperature
        
        # Sample from probability distribution
        probs = softmax(next_logits)
        next_idx = np.random.choice(len(probs), p=probs)
        
        indices.append(next_idx)
    
    return decode(indices)

print("Generation function defined.")

## 5. Training Loop

In [ ]:
# Hyperparameters
d_model = 64
n_heads = 4
d_ff = 256
n_layers = 3
max_seq_len = 32
batch_size = 32
learning_rate = 1e-3
n_steps = 3000

# Create model
np.random.seed(42)
model = Transformer(vocab_size, d_model, n_heads, d_ff, n_layers, max_seq_len)
optimizer = Adam(model.get_params(), lr=learning_rate)

print(f"Model parameters: {model.count_params():,}")
print(f"Training for {n_steps} steps...")
print(f"Batch size: {batch_size}, Sequence length: {max_seq_len}")
print()

In [ ]:
# Generate text before training (should be random garbage)
print("=== Before Training ===")
print(generate(model, "First", max_len=100, temperature=1.0))
print()

In [ ]:
# Training loop
losses = []
start_time = time.time()

for step in range(n_steps):
    # Get batch
    x_batch, y_batch = get_batch(data, batch_size, max_seq_len)
    
    # Forward pass
    logits = model.forward(x_batch)  # (B, T, vocab_size)
    
    # Compute loss
    logits_flat = logits.reshape(-1, vocab_size)
    targets_flat = y_batch.reshape(-1)
    loss, dlogits_flat = cross_entropy_loss(logits_flat, targets_flat)
    losses.append(loss)
    
    # Backward pass
    dlogits = dlogits_flat.reshape(batch_size, max_seq_len, vocab_size)
    model.backward(dlogits)
    
    # Update parameters
    optimizer.step()
    
    # Print progress
    if (step + 1) % 200 == 0:
        avg_loss = np.mean(losses[-200:])
        elapsed = time.time() - start_time
        steps_per_sec = (step + 1) / elapsed
        print(f"Step {step+1:4d}/{n_steps} | Loss: {avg_loss:.4f} | {steps_per_sec:.1f} steps/s")

total_time = time.time() - start_time
print(f"\nTraining complete in {total_time:.1f}s")
print(f"Final average loss: {np.mean(losses[-200:]):.4f}")

## 6. Training Loss Plot

In [ ]:
# Plot training loss
plt.figure(figsize=(10, 4))

# Smooth the loss for better visualization
window = 50
smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')

plt.plot(losses, alpha=0.2, color='blue', label='Raw loss')
plt.plot(range(window-1, len(losses)), smoothed, color='blue', linewidth=2, label=f'Smoothed (window={window})')
plt.axhline(y=np.log(vocab_size), color='red', linestyle='--', alpha=0.5, label=f'Random baseline (ln({vocab_size})={np.log(vocab_size):.2f})')

plt.xlabel('Training Step')
plt.ylabel('Cross-Entropy Loss')
plt.title('Training Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Text Generation

In [ ]:
# Generate text after training
print("=== After Training ===")
print()

prompts = ["First", "MENENIUS", "The ", "What "]
for prompt in prompts:
    print(f"--- Prompt: '{prompt}' ---")
    print(generate(model, prompt, max_len=200, temperature=0.8))
    print()

In [ ]:
# Temperature comparison
print("=== Temperature Comparison ===")
print()
for temp in [0.3, 0.8, 1.5]:
    print(f"--- Temperature: {temp} ---")
    print(generate(model, "First Citizen", max_len=150, temperature=temp))
    print()

## 8. Visualize Learned Attention Patterns

In [ ]:
# Feed a sample through and visualize attention in the first block
sample_text = "First Citizen:\nBefore we pr"
sample_indices = np.array([encode(sample_text)])

_ = model.forward(sample_indices)

# Visualize attention heads from the first transformer block
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
chars_display = list(sample_text)
T = len(chars_display)

for h in range(4):
    attn = model.blocks[0].mha.attn_weights[0, h, :T, :T]
    im = axes[h].imshow(attn, cmap='hot', vmin=0)
    axes[h].set_title(f'Head {h+1}')
    
    # Show character labels (every 4th character for readability)
    tick_positions = range(0, T, max(1, T//8))
    tick_labels = [chars_display[i] if chars_display[i] != '\n' else '\\n' for i in tick_positions]
    axes[h].set_xticks(list(tick_positions))
    axes[h].set_xticklabels(tick_labels, fontsize=7)
    axes[h].set_yticks(list(tick_positions))
    axes[h].set_yticklabels(tick_labels, fontsize=7)

plt.suptitle('Learned Attention Patterns (Block 1)', y=1.02)
plt.tight_layout()
plt.show()

## Summary

We built a **complete, trainable decoder-only transformer from scratch** using only NumPy:

- **Character-level tokenization**: no external tokenizer needed
- **Learnable embeddings** + sinusoidal positional encoding
- **Multi-head self-attention** with causal masking
- **Feed-forward network** with ReLU activation
- **Layer normalization** and **residual connections**
- **Adam optimizer** with gradient clipping
- **Autoregressive text generation** with temperature sampling

### Key observations:
1. **Loss decreased** from ~ln(vocab_size) to a much lower value, proving the model is learning
2. **Generated text** shows English character patterns (words, spacing, punctuation)
3. **Lower temperature** produces more repetitive but coherent text
4. **Higher temperature** produces more varied but less coherent text
5. Attention heads learn **different patterns** (some attend locally, others attend to specific tokens)

### What we achieved vs. the reference repo:
| Feature | Reference | Ours |
|---------|-----------|------|
| Training | None | Full training with backprop |
| Embeddings | GloVe (external) | Learned from scratch |
| Output | Random noise | Learned character patterns |
| Dependencies | numpy, gensim | numpy only (+ matplotlib for plots) |